# Stage 1: the ODE as an `OrdinaryDifferentialSystem`

An `OrdinaryDifferentialSystem` holds the right-hand side of the equation  $\dot{u}_i = f_i(u)$ and the index ranges that fix the grid.
| constructor | takes |
|---|---|
| `from_sympy` | a SymPy expression |
| `from_strings` | one right-hand-side string, parsed with the index names in scope |

It is suggested to use either of these constructors rather than constructing with `__init__`. $u$ is an `sympy.IndexedBase`, and $i$ is a bounded `Idx`.

In [1]:
import sympy as sp

from lin_ode import DirichletCondition, Facet, OrdinaryDifferentialSystem

D, Q, R, h = sp.symbols("D Q R h")
u = sp.IndexedBase("u")            # the state; must NOT carry a shape, the indices supply it
i = sp.Idx("i", (0, 3))            # 4 sites, inclusive bounds

ode = OrdinaryDifferentialSystem.from_sympy(
    D * (u[i - 1] - 2 * u[i] + u[i + 1]) / h**2 + Q * u[i] + R * u[i] ** 2, u, (i,)
)
ode.equation.expr

D*(u[Mod(i + 1, 4)] + u[Mod(i + 3, 4)] - 2*u[i])/h**2 + Q*u[i] + R*u[i]**2

## Properties of an `OrdinaryDifferentialSystem`

Every property is read-only. The ones below describe the grid. `equation` holds the right-hand side
as a nested `IndexedExpression`.

In [2]:
print("state_variable  ", ode.state_variable)
print("index_symbols   ", ode.index_symbols, " bounds:", [(x.lower, x.upper) for x in ode.index_symbols])
print("shape           ", ode.shape, "   <- one entry per index")
print("total_dof       ", ode.total_dof, "     <- product of the shape")
print("equation.indices", ode.equation.indices)
print("equation.size   ", ode.equation.size)

state_variable   u
index_symbols    (i,)  bounds: [(0, 3)]
shape            (4,)    <- one entry per index
total_dof        4      <- product of the shape
equation.indices (i,)
equation.size    4


`shape` and `total_dof` come from the index bounds. A state variable that carried its own shape
would be a second source of truth, so the constructor rejects it. The bounds must be concrete
integers, for the same reason.

## Boundary conditions

The index space is periodic by default, so an edge with no condition wraps. The wrap stays inside
the index's own bounds: an `Idx` running from $a$ to $b$ maps $i$ to $a+(i-a)\bmod n$, with
$n=b-a+1$ sites. For an index that starts at 0 that is $i$ modulo $n$.

To pin an edge instead, pass a `DirichletCondition` through `conditions`. The right-hand side you pass still applies at
every other site, and each condition becomes its own branch of the assembled equation.

[`07_boundary_conditions`](07_boundary_conditions.ipynb) covers the rest: which region a condition
can name, values that depend on the state, conditions that set the derivative instead, and which
of two overlapping conditions wins.

## `from_strings`

`from_strings` builds the same object as `from_sympy`, from a string instead of an expression. It
parses the string with the index names in scope, so you declare no symbols first. The state
variable is `u` unless you name another one.

```python
# these two build the same OrdinaryDifferentialSystem
OrdinaryDifferentialSystem.from_sympy(D * u[i], u, (i,))
OrdinaryDifferentialSystem.from_strings("D * u[i]", indices=(i,))
```

Reach for it when the expression arrives as text. Otherwise prefer `from_sympy`, for the reason
below.

### Names SymPy has already taken

Every other bare name goes to SymPy's own namespace, and anything defined there wins. The name you
meant never becomes a symbol.

| you write | what you get |
| --- | --- |
| `N`, `Q`, `S`, `O`, `beta` | a `TypeError`. Each one is a SymPy object |
| `E`, `I`, `pi` | **silently**, Euler's number, the imaginary unit and $\pi$ |

The silent group is the dangerous one. It raises no error and gives you the wrong expression. $Q$
is a KPP–Fisher coefficient that `../examples/` uses throughout, so that clash happens in practice.

The table lists what you are most likely to hit, not every name at risk. Rather than memorise one,
use `from_sympy` with explicit `sp.symbols` when a name matters. That route never consults the
SymPy namespace.

```python
Q = sp.symbols("Q")
OrdinaryDifferentialSystem.from_sympy(Q * u[i], u, (i,))
```

## Multi-index grids

Pass several indices and the state takes one subscript per index, in that order. `(i, j)` means
`u[i, j]`, with `i` first.

In [3]:
i = sp.Idx("i", (0, 5))
j = sp.Idx("j", (0, 7))
plate = OrdinaryDifferentialSystem.from_strings(
    "(u[i - 1, j] + u[i + 1, j] + u[i, j - 1] + u[i, j + 1] - 4 * u[i, j]) / h**2",
    indices=(i, j),
    conditions=[
        DirichletCondition(Facet.low(i), 0),
        DirichletCondition(Facet.low(j), 0),
    ],
)
print("shape    ", plate.shape, "  <- (i, j), in that order")
print("total_dof", plate.total_dof)
print("fallback ", plate.equation.cases[-1][0])

shape     (6, 8)   <- (i, j), in that order
total_dof 48
fallback  (u[Mod(i + 1, 6), j] + u[Mod(i + 5, 6), j] + u[i, Mod(j + 1, 8)] + u[i, Mod(j + 7, 8)] - 4*u[i, j])/h**2


## Input validation

`from_sympy` and `from_strings` both reject bad input at construction, rather than let it surface
later as a wrong matrix element. The common mistakes:

- **A state variable created with an explicit shape.** `shape` comes from the index bounds, so a
  second source of truth is an error.
- **An index with no bounds.** Write `sp.Idx("i", (0, 3))`, never a bare `sp.Idx("i")`.
- **Subscripts in inconsistent order.** `u[i, j] + u[j, i]` indexes the same state two ways.
- **No indices at all.** A system needs at least one.

Each raises `ValueError` with the offending symbol named.

You may use literal indices. A boundary branch names a site that way, as `u[0]` or `u[3]`. Only the
symbolic subscripts must agree on their order.

Write the number, not a name. A bare `N` is one of the names SymPy has already taken.

---

[← Overview](01_overview.ipynb) · [Notebook index](../README.md) · [2. Linearization →](03_linearization.ipynb)